# 🧠 Aula 09 — Introdução às Redes Neurais Artificiais

**Disciplina:** Inteligência Artificial Aplicada à Engenharia Química  
**Dataset:** `coluna_destilacao_30dias.csv` (8 variáveis + composição)

---

## Contexto

Até agora usamos Random Forest e XGBoost (modelos baseados em árvores). Eles dividem o espaço em regiões retangulares ("degraus"). E se a relação entre features e target for uma **curva suave e complexa**? É aqui que entram as redes neurais.

## 3.1 — Exercício Guiado: MLP vs XGBoost

Compare uma **MLP** com o **XGBoost** no dataset da coluna de destilação.

### Passo 1: Carregar dados + features

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPRegressor
from xgboost import XGBRegressor
from sklearn.metrics import r2_score, mean_squared_error
import matplotlib.pyplot as plt

URL = "https://raw.githubusercontent.com/LuisGSVasconcelos/IA_EngQuimica/main/dados/aula04/coluna_destilacao_30dias.csv"
df = pd.read_csv(URL, parse_dates=['timestamp'])
df.set_index('timestamp', inplace=True)

# Features (mesmas da Aula 8: lags + média móvel)
feature_cols = ['T_top_C', 'T_base_C', 'P_coluna_kPa', 'R_refluxo', 'F_alimentacao_kg_h']
df['T_top_lag1'] = df['T_top_C'].shift(1)
df['T_top_lag3'] = df['T_top_C'].shift(3)
df['R_ma5'] = df['R_refluxo'].rolling(5).mean()
df.dropna(inplace=True)

X = df[feature_cols + ['T_top_lag1', 'T_top_lag3', 'R_ma5']]
y = df['composicao_destilado']
print(f"Features: {X.shape[1]}  |  Amostras: {X.shape[0]}")

### Passo 2: Split + Escalonamento (CRÍTICO para MLP)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Redes neurais exigem dados normalizados!
# IMPORTANTE: fit no treino, transform no teste
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
print("Dados escalados (média=0, desvio=1)")

### Passo 3: Treinar MLP

In [ ]:
mlp = MLPRegressor(
    hidden_layer_sizes=(64, 32),
    activation='relu',
    solver='adam',
    max_iter=500,
    random_state=42,
    early_stopping=True,
    validation_fraction=0.1
)

import time
t0 = time.time()
mlp.fit(X_train_scaled, y_train)
tempo_mlp = time.time() - t0

y_pred_mlp = mlp.predict(X_test_scaled)
rmse_mlp = np.sqrt(mean_squared_error(y_test, y_pred_mlp))
r2_mlp = r2_score(y_test, y_pred_mlp)
print(f"MLP: RMSE={rmse_mlp:.4f}  R²={r2_mlp:.4f}  Tempo={tempo_mlp:.1f}s")

### Passo 4: Treinar XGBoost (baseline, sem escala)

In [ ]:
xgb = XGBRegressor(n_estimators=200, learning_rate=0.1, random_state=42, verbosity=0)
t0 = time.time()
xgb.fit(X_train, y_train)
tempo_xgb = time.time() - t0

y_pred_xgb = xgb.predict(X_test)
rmse_xgb = np.sqrt(mean_squared_error(y_test, y_pred_xgb))
r2_xgb = r2_score(y_test, y_pred_xgb)
print(f"XGBoost: RMSE={rmse_xgb:.4f}  R²={r2_xgb:.4f}  Tempo={tempo_xgb:.1f}s")

### Passo 5: Comparar

In [ ]:
print(f"{'Modelo':<20} {'RMSE':>8} {'R²':>8} {'Tempo':>8}")
print('-' * 44)
print(f"{'MLP (64,32)':<20} {rmse_mlp:>8.3f} {r2_mlp:>8.3f} {tempo_mlp:>6.1f}s")
print(f"{'XGBoost (200)':<20} {rmse_xgb:>8.3f} {r2_xgb:>8.3f} {tempo_xgb:>6.1f}s")

print("\nA MLP superou o XGBoost? Precisão? Velocidade?")

### Passo 6: Plot comparativo

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].scatter(y_test, y_pred_mlp, alpha=0.3, s=8)
axes[0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
axes[0].set_title(f'MLP — RMSE: {rmse_mlp:.3f}')
axes[0].set_xlabel('Real'); axes[0].set_ylabel('Predito')

axes[1].scatter(y_test, y_pred_xgb, alpha=0.3, s=8, color='green')
axes[1].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
axes[1].set_title(f'XGBoost — RMSE: {rmse_xgb:.3f}')
axes[1].set_xlabel('Real'); axes[1].set_ylabel('Predito')
plt.tight_layout()
plt.show()

### ✏️ Pausa reflexiva (2 min)

A MLP com arquitetura (64, 32) tem quantos parâmetros no total (pesos + biases)?

- Camada 1: 8 features → 64 neurônios = 8×64 + 64 = 576
- Camada 2: 64 → 32 = 64×32 + 32 = 2080
- Saída: 32 → 1 = 32 + 1 = 33
- **Total: 576 + 2080 + 33 = 2689 parâmetros**

> _Escreva seu cálculo aqui..._

---

## 3.2 — Exercício em Grupo: Arquitetura da MLP

Cada grupo testa uma arquitetura e função de ativação diferente.

| Grupo | hidden_layer_sizes | activation |
|-------|-------------------|------------|
| **A** | (16,) | relu |
| **B** | (64, 32) | relu |
| **C** | (256, 128) | relu |
| **D** | (64, 32) | tanh |

In [ ]:
# Teste a arquitetura do seu grupo
arquitetura = (64, 32)   # ← Mude para a do seu grupo
ativacao = 'relu'        # ← 'relu' ou 'tanh'

mlp_grupo = MLPRegressor(
    hidden_layer_sizes=arquitetura,
    activation=ativacao,
    max_iter=500, random_state=42,
    early_stopping=True
)
mlp_grupo.fit(X_train_scaled, y_train)

rmse_tr = np.sqrt(mean_squared_error(y_train, mlp_grupo.predict(X_train_scaled)))
rmse_te = np.sqrt(mean_squared_error(y_test, mlp_grupo.predict(X_test_scaled)))
print(f"Arquitetura {arquitetura} ({ativacao}):")
print(f"  RMSE treino: {rmse_tr:.4f}")
print(f"  RMSE teste:  {rmse_te:.4f}  (gap={rmse_te-rmse_tr:.4f})")

# Curva de perda (convergiu?)
plt.figure(figsize=(8, 4))
plt.plot(mlp_grupo.loss_curve_)
plt.xlabel('Iteração')
plt.ylabel('Perda')
plt.title(f'Curva de Perda — {arquitetura} ({ativacao})')
plt.grid(alpha=0.3)
plt.show()

> **Perguntas para o grupo:**
> 1. A arquitetura maior (mais parâmetros) sempre melhora o RMSE de teste?
> 2. O treino convergiu? (olhe a curva de perda)
> 3. `tanh` vs `relu` — qual converge mais rápido? A ativação muda o RMSE?

### 🧠 Desafio extra (NT)

Por que a MLP com (256, 128, 64) pode ter RMSE de teste PIOR que a (64, 32) mesmo com RMSE de treino menor?

> _Overfitting + maldição da dimensionalidade dos parâmetros..._

---

## Checklist de Arquitetura

- [ ] Dados escalados (StandardScaler)
- [ ] Arquitetura definida
- [ ] Ativação escolhida
- [ ] Treino convergiu? (curva de perda)
- [ ] RMSE calculado
- [ ] Overfitting diagnosticado